E-22: the Lalonde job-training data (illustration, ATT of the NSW treated)

Registration: `doc/2026-10-07_experiment_registration.md` (parent repository), E-22, and §1.1–§1.7; the design document is not binding (U28) and the open choices are in `grrexp.e22.CHOICES`. One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-22 stage0.5 19_E22_lalonde.ipynb`
- `... E-22 stage1 ...` (after the pilot is committed).

The data, the arms, the replication function over one fold split, the full-sample fits and the aggregation are in `grrexp/e22.py`. A solver that raises stops the notebook and the run is recorded as failed (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import baselines, e22, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
if STAGE == "stage0":
    raise RuntimeError("E-22 has no Stage 0 (no population predictions)")
rec = outputs.RunRecorder(22, STAGE, cells=e22.CELLS, arms=[e22.record_label(a) for a in e22.ARM_LABELS])
#: Stage 1 workers (results do not depend on the number of workers; Stage 0.5 checks 1 vs 12)
STAGE1_WORKERS = 6
STAGE

'stage0.5'

# The workflow after the fold splits (shared by Stage 0.5 and Stage 1)

The full-sample fits (weights for the SMD (a), the EB check against the independent dual Newton, the admissibility diagnostic) and the split-median aggregation per arm. The EB check stops the notebook if UKL (C = 0) x D1 and the independent dual Newton differ by more than 1e-6 (relative).

In [2]:
def aggregate(raw, entropy):
    full, warns, _ = baselines.run_recording_warnings(lambda: e22.full_sample(entropy))
    if not full["eb_check"]["agrees"]:
        raise RuntimeError(f"EB check failed: {full['eb_check']}")
    summ = e22.summarise(raw, full)
    summ["full_sample_warnings"] = json.dumps(warns)
    return summ, int(sum(warns.values())) + int(full["n_warnings"])

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

In [3]:
if STAGE == "stage0.5":
    parallel.configure_worker()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e22.CELLS):
            t0 = time.perf_counter()
            tasks = e22.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, cells=[ci])
            res = parallel.run_tasks(e22.replicate, tasks, workers)
            aggregate(e22.raw_frame(tasks, res), PILOT_ENTROPY)
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e22.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    summ, full_warn = aggregate(raw, PILOT_ENTROPY)
    summ.to_csv(buf, index=False)
    buf.seek(0)
    S = pd.read_csv(buf)
    e22.tables(S)
    whole_seconds = time.perf_counter() - t0
    counts = raw.groupby(["cell", "arm", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e22.CELLS[c] for c in counts["cell"]],
                              "arm": counts["arm"].map(e22.record_label),
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e22.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    arw_like = ~raw["arm"].str.endswith("|TMLE")  # one record per arm and split
    rec.finalize(R={"main": outputs.PILOT_REPS}, n={"main": [e22.N]},
                 warnings=e22.total_warnings(raw) + full_warn,
                 failures=int((arw_like & (raw["status"] != "ok")).sum()),
                 extra={"whole_pilot_aggregation_and_table_seconds": whole_seconds})
    del raw

workers
1     225.331178
12     43.311003
Name: seconds, dtype: float64 1.2676918329962064
arm          status                        
BKL-D1_ARW   domain_prediction                 10
BKL-D1_TMLE  domain_prediction                 10
BKL-D2_ARW   domain_prediction                 10
BKL-D2_TMLE  domain_prediction                 10
BKL-D3_ARW   cv_failed                         10
BKL-D3_TMLE  cv_failed                         10
BP-D1_ARW    uncertified_numerical_boundary    10
BP-D1_TMLE   uncertified_numerical_boundary    10
BP-D2_ARW    uncertified_numerical_boundary    10
BP-D2_TMLE   uncertified_numerical_boundary    10
BP-D3_ARW    ok                                10
BP-D3_TMLE   ok                                10
DML-GBM      ok                                10
LogitAIPW    ok                                10
SQ-D1_ARW    ok                                10
SQ-D1_TMLE   ok                                10
SQ-D2_ARW    ok                                10
SQ-D2_TMLE   ok

# Stage 1: the 100 fold splits

In [4]:
if STAGE == "stage1":
    parallel.configure_worker()
    tasks = e22.tasks_for(CONFIRMATORY_ENTROPY, e22.R)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e22.replicate, tasks, STAGE1_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw = e22.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e22.total_warnings(raw)
    raw.shape, n_warn

In [5]:
if STAGE == "stage1":
    summ, full_warn = aggregate(raw, CONFIRMATORY_ENTROPY)
    rec.write_summary(summ)
    {k: json.loads(summ[k].iloc[0]) for k in ("eb_check", "admissibility")}

# Tables and figure (from summary.csv only, §4)

In [6]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    for name, tex in e22.tables(S).items():
        rec.write_table(name, tex)
    rec.save_figure(e22.figure(S), "fig_E22_love")
    print(S[["arm", "R_s", "estimate", "se", "full_smd_raw_max", "full_ess_c"]].to_string())
    arw_like = ~S["arm"].str.endswith("|TMLE")
    rec.finalize(R={"main": e22.R}, n={"main": [e22.N]}, warnings=int(n_warn + full_warn),
                 failures=int((S.loc[arw_like, "R"] - S.loc[arw_like, "R_s"]).sum()))